# Before/After Displacement on Equilateral-Triangle Plane

- Before points: red
- After points: blue
- Displacement: arrows from before to after
- Grid: 0.1 spacing on barycentric coordinates


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from coordinate_fileio import load_division_result
from sphere_division_algorithms import build_octant_mesh, project_vertex
from sphere_index_util import iter_valid_ij


In [ ]:
V_I = np.array([0.0, 0.0], dtype=float)
V_J = np.array([1.0, 0.0], dtype=float)
V_K = np.array([0.5, np.sqrt(3.0) / 2.0], dtype=float)


def xyz_to_barycentric(xyz):
    xyz = np.asarray(xyz, dtype=float)
    s = float(np.sum(xyz))
    if s <= 0.0:
        raise ValueError('sum(x,y,z) must be positive.')
    return xyz / s


def barycentric_to_uv(w):
    return w[0] * V_I + w[1] * V_J + w[2] * V_K


def xyz_to_uv(xyz):
    return barycentric_to_uv(xyz_to_barycentric(xyz))


def draw_barycentric_grid(ax, step=0.1, color='0.85', lw=0.8):
    levels = np.arange(step, 1.0, step)

    # i/(i+j+k) = t
    for t in levels:
        p0 = barycentric_to_uv(np.array([t, 0.0, 1.0 - t]))
        p1 = barycentric_to_uv(np.array([t, 1.0 - t, 0.0]))
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=color, lw=lw, zorder=0)

    # j/(i+j+k) = t
    for t in levels:
        p0 = barycentric_to_uv(np.array([0.0, t, 1.0 - t]))
        p1 = barycentric_to_uv(np.array([1.0 - t, t, 0.0]))
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=color, lw=lw, zorder=0)

    # k/(i+j+k) = t
    for t in levels:
        p0 = barycentric_to_uv(np.array([0.0, 1.0 - t, t]))
        p1 = barycentric_to_uv(np.array([1.0 - t, 0.0, t]))
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=color, lw=lw, zorder=0)


In [ ]:
result_path = Path('results/division_result_16.json')

N, positions_after = load_division_result(result_path)
points_before_raw, _, _ = build_octant_mesh(N)
positions_before = np.full_like(points_before_raw, np.nan)
for i, j in iter_valid_ij(N):
    positions_before[i, j] = project_vertex(points_before_raw[i, j], (i, j), N)

uv_before = []
uv_after = []
for i, j in iter_valid_ij(N):
    if np.isnan(positions_after[i, j]).any():
        continue
    uv_before.append(xyz_to_uv(positions_before[i, j]))
    uv_after.append(xyz_to_uv(positions_after[i, j]))

uv_before = np.asarray(uv_before, dtype=float)
uv_after = np.asarray(uv_after, dtype=float)
disp = uv_after - uv_before
disp_len = np.linalg.norm(disp, axis=1)

print(f'loaded: {result_path.as_posix()}')
print(f'N={N}, point_count={uv_before.shape[0]}')
print(f'displacement length: min={disp_len.min():.6e}, mean={disp_len.mean():.6e}, max={disp_len.max():.6e}')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

# triangle boundary
tri = np.vstack([V_I, V_J, V_K, V_I])
ax.plot(tri[:, 0], tri[:, 1], color='black', lw=1.4, zorder=2)

# 0.1 grid
draw_barycentric_grid(ax, step=0.1, color='0.85', lw=0.8)

# displacement arrows
ax.quiver(
    uv_before[:, 0],
    uv_before[:, 1],
    disp[:, 0],
    disp[:, 1],
    angles='xy',
    scale_units='xy',
    scale=1.0,
    color='0.35',
    width=0.0020,
    alpha=0.7,
    zorder=1,
)

# points
ax.scatter(uv_before[:, 0], uv_before[:, 1], s=20, c='red', label='before', alpha=0.85, zorder=3)
ax.scatter(uv_after[:, 0], uv_after[:, 1], s=20, c='blue', label='after', alpha=0.85, zorder=4)

# vertex labels
ax.text(V_I[0] - 0.05, V_I[1] - 0.04, '(1, 0, 0)', fontsize=10)
ax.text(V_J[0] + 0.01, V_J[1] - 0.04, '(0, 1, 0)', fontsize=10)
ax.text(V_K[0] - 0.02, V_K[1] + 0.03, '(0, 0, 1)', fontsize=10)

ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-0.08, 1.08)
ax.set_ylim(-0.08, np.sqrt(3.0) / 2.0 + 0.08)
ax.set_xlabel('u')
ax.set_ylabel('v')
ax.set_title(f'Projected displacement on equilateral-triangle plane (N={N})')
ax.legend(loc='upper right')
ax.grid(False)
plt.tight_layout()
plt.show()
